# 04 — Feature Engineering

Engineers borrower-level risk features on top of the cleaned data: income/repayment ratios, credit-risk
buckets, and employment/credit-history signals. All feature logic lives in
`credit_risk.features.engineering` — this notebook calls `build_features()` and inspects the result.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from credit_risk.config import settings
from credit_risk.data import clean_and_label, load_raw_data
from credit_risk.features import build_features

In [ ]:
df = build_features(clean_and_label(load_raw_data(settings.raw_data_path)))
print(df.shape)

## Engineered features

| Feature | What it captures |
|---|---|
| `income_to_loan_ratio` | annual income / loan amount |
| `revol_util_bucket` | revolving utilization, bucketed |
| `dti_bucket` | debt-to-income, bucketed |
| `employment_stability` | parsed `emp_length`, bucketed |
| `credit_history_years` | years between earliest credit line and loan issuance |
| `inquiry_risk_bucket` | recent credit inquiries, bucketed |
| `delinquency_risk` | 2-year delinquency count, bucketed |
| `loan_size_bucket` | loan amount quartile |
| `revol_bal_to_income` | revolving balance / annual income |
| `installment_to_income` | annualized installment / annual income |
| `credit_exposure_score` | `revol_util + dti + inq_last_6mths`, NaN-safe |

`loan_size_bucket` uses `pd.qcut`, whose bin edges are computed from whatever DataFrame is passed in — a
real train/inference skew risk worth revisiting before this goes into production.

In [ ]:
new_columns = [
    "income_to_loan_ratio", "revol_util_bucket", "dti_bucket", "employment_stability",
    "credit_history_years", "inquiry_risk_bucket", "delinquency_risk", "loan_size_bucket",
    "revol_bal_to_income", "installment_to_income", "credit_exposure_score",
]
df[new_columns].head()

## DTI bucket vs. default

In [ ]:
df.groupby("dti_bucket", observed=True)["target_default"].mean() * 100

## Employment stability vs. default

In [ ]:
df.groupby("employment_stability", observed=True)["target_default"].mean() * 100

## Inquiry risk vs. default

In [ ]:
df.groupby("inquiry_risk_bucket", observed=True)["target_default"].mean() * 100

**Findings:** DTI risk is monotonic — each bucket defaults more than the last. Employment stability shows a
modest gradient (newer employment = somewhat higher risk). Inquiry risk is *not* monotonic —
`very_high_inquiry` defaults less than `high_inquiry`, likely because very recent multi-inquiry shoppers
skew toward rate-shopping prime borrowers rather than credit-seeking distressed ones. Worth a closer look
later, not an explanation to trust blindly.

## Credit exposure score distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["credit_exposure_score"], bins=50)
plt.title("Credit Exposure Score Distribution")
plt.show()